# 03 · Check the Blender/Bonsai edit

`models/blender_edit.ifc` was saved from Bonsai over MCP. It has two edits to `AC20-FZK-Haus.ifc`:

1. `Pset_WallCommon.ThermalTransmittance` on `Wand-Ext-ERDG-1` set from 0.4 to 0.28.
2. The gable roof replaced by a barrel arc, with its rafters, purlins, upper walls and Galerie adapted. These are geometry changes only.

This notebook opens both files with the venv's ifcopenshell, diffs every wall's psets, and asserts that the thermal value is the only property change. It doesn't check the roof geometry (see `docs/fzk-haus-bonsai-edits.md`).

In [ ]:
import ifcopenshell, ifcopenshell.util.element as ue
print(ifcopenshell.version)
orig = ifcopenshell.open('../models/AC20-FZK-Haus.ifc')
edit = ifcopenshell.open('../models/blender_edit.ifc')
print(orig.schema, edit.schema, len(orig.by_type('IfcWall')), len(edit.by_type('IfcWall')))

In [ ]:
def flat(f):
    out = {}
    for w in f.by_type('IfcWall'):
        for pset, props in ue.get_psets(w).items():
            for k, v in props.items():
                if k != 'id':
                    out[(w.GlobalId, w.Name, pset, k)] = v
    return out

a, b = flat(orig), flat(edit)
changed = {k: (a.get(k), b.get(k)) for k in a.keys() | b.keys() if a.get(k) != b.get(k)}
print(len(a), 'properties compared')
for k, v in changed.items():
    print(k, v)

In [ ]:
assert changed == {('3rPX_Juz59peXXY6wDJl18', 'Wand-Ext-ERDG-1', 'Pset_WallCommon', 'ThermalTransmittance'): (0.4, 0.28)}
print('OK: exactly one property changed')